In [1]:
library(dplyr)
library(quanteda)
library(stringr)
library(data.table)
library(maps)
library(lubridate)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Package version: 4.3.1
Unicode version: 15.0
ICU version: 73.2

Parallel computing: disabled

See https://quanteda.io for tutorials and examples.


Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last


Warning message:
“‘timedatectl’ indicates the non-existent timezone name ‘n/a’”
Warning message:
“Your system is mis-configured: ‘/etc/localtime’ is not a symlink”
Warning message:
“It is strongly recommended to set envionment variable TZ to ‘Etc/UCT’ (or equivalent)”

Attaching package: ‘lubridate’


The following objects are masked from ‘package:data.table’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


The following objects are masked from ‘package:base’:

    date, intersect, setdiff,

## Remove non-US articles

In [2]:
all_df <- readRDS("all_df_20260406.rds")
df_wp_historical <- readRDS("df_wp_clean_OCR.rds")
df_wsj_historical <- readRDS("df_wsj_clean_OCR.rds")

In [3]:
colnames(df_wp_historical)

[1] "Article_ID"         "Title"              "Date"              
 [4] "Abstract"           "Text"               "Source"            
 [7] "Location"           "People"             "Organization"      
[10] "Type"               "Desk"               "percentage_covered"

In [4]:
nrow(all_df)
nrow(df_wp_historical)
nrow(df_wsj_historical)

[1] 9305389

[1] 488904

[1] 146692

In [5]:
### Screen out low OCR quality texts
df_wp_historical <- 
    df_wp_historical %>%
    filter(percentage_covered >= 80) %>%
    select(-percentage_covered)

df_wsj_historical <- 
    df_wsj_historical %>%
    filter(percentage_covered >= 80) %>%
    select(-percentage_covered)

In [6]:
all_df <- bind_rows(all_df, df_wp_historical, df_wsj_historical)

In [7]:
nrow(all_df)

[1] 9895568

In [15]:
# --- A. Prepare Non-U.S. Keywords ---
non_us_csv <- read.csv("non_us_country_adj.csv")
# 1. Combine Country, Adjective, and Demonym from file
non_us_base <- unique(c(non_us_csv$Country, non_us_csv$Adjectival, non_us_csv$Demonym))
non_us_base <- non_us_base[!is.na(non_us_base) & non_us_base != ""]

# 2. Additional non-US names identified
additional_exclusions <- c(
  "Europe", "Asia", "Africa", "South America", "Oceania", "Atlantic Ocean", "North Sea", "North Pole", "Pacific Ocean",
  "Middle East",  "Latin America", "Central America", "Southeast Asia", "Caribbean", "Arctic", "Antarctica",
  "USSR", "Soviet Union", "Soviet", "union of soviet socialist republics", "Siberia",
  "Great Britain", "United Kingdom", "UK", "Ireland", "Dominican Republic", "Zaire", "Bosnia-Hercegovina", "Zimababwe",
  "Burma", "Tibet", "Bahamas", "Macao", "Mount Everest", "Sicily", "Kashmir",
  "Lithuania", "Rumania", "Yugoslavia", "Czechoslovakia", "East Germany", "West Germany",
  "European Union", "EU", "OPEC"
)

# Add major international cities (excluding USA)
data(world.cities)
intl_cities <- world.cities[world.cities$country.etc != "USA" & world.cities$pop > 500000, "name"]
non_us_list <- unique(c(non_us_base, intl_cities, additional_exclusions))

# Create Non-U.S. Regex Pattern
non_us_pattern <- paste0("\\b(", paste(non_us_list, collapse = "|"), ")\\b")

In [16]:
# --- B. Prepare U.S. Keywords ---
us_states <- state.name
us_markers <- c("U\\.S\\.", "U\\.S\\.A\\.", "USA", "United States", "American", us_states)
us_pattern <- paste0("\\b(", paste(us_markers, collapse = "|"), ")\\b")

In [17]:
us_pattern

[1] "\\b(U\\.S\\.|U\\.S\\.A\\.|USA|United States|American|Alabama|Alaska|Arizona|Arkansas|California|Colorado|Connecticut|Delaware|Florida|Georgia|Hawaii|Idaho|Illinois|Indiana|Iowa|Kansas|Kentucky|Louisiana|Maine|Maryland|Massachusetts|Michigan|Minnesota|Mississippi|Missouri|Montana|Nebraska|Nevada|New Hampshire|New Jersey|New Mexico|New York|North Carolina|North Dakota|Ohio|Oklahoma|Oregon|Pennsylvania|Rhode Island|South Carolina|South Dakota|Tennessee|Texas|Utah|Vermont|Virginia|Washington|West Virginia|Wisconsin|Wyoming)\\b"

### STEP 1: Articles WITH Location Tags

In [11]:
# Identify articles with tags
df_with_tags <- all_df[!is.na(Location) & Location != ""]

In [12]:
nrow(df_with_tags)

[1] 3110058

In [18]:
df_with_tags_filtered <-
  df_with_tags %>%
  mutate(
    # Check for presence of US and Non-US keywords in the raw string
    has_us = str_detect(Location, regex(us_pattern, ignore_case = TRUE)),
    has_non_us = str_detect(Location, regex(non_us_pattern, ignore_case = TRUE))
  ) %>%
  # Logic: Keep if it has a US tag OR (it doesn't have a non-US tag)
  # This automatically removes rows that have non-US tags but NO US tags.
  filter(has_us | !has_non_us) %>%
  select(-has_us, -has_non_us)

In [16]:
nrow(df_with_tags_filtered)

[1] 2265611

In [17]:
location <-
  df_with_tags_filtered %>%
  mutate(Location = tolower(Location)) %>%
  group_by(Location) %>%
  summarise(Count = n()) %>%
  ungroup() %>%
  arrange(desc(Count)) %>%
  filter(Count >= 300)

In [18]:
write.csv(location, "location.csv")

### STEP 2: Untagged Articles

In [20]:
df_no_tags <- 
  all_df %>%
  filter(is.na(Location) | Location == "")

In [20]:
nrow(df_no_tags)

[1] 6785510

In [22]:
df_no_tags_filtered <- 
  df_no_tags %>%
  filter(!(
    # 1. Title contains non-U.S. location
    str_detect(Title, regex(non_us_pattern, ignore_case = TRUE)) & 
    # 2. Title does NOT mention U.S. locations
    !str_detect(Title, regex(us_pattern, ignore_case = TRUE))
  ))

In [22]:
nrow(df_no_tags_filtered)

[1] 6384745

### Combine & deduplicate

In [24]:
final_dataset <- bind_rows(df_with_tags_filtered, df_no_tags_filtered)

In [7]:
nrow(final_dataset)

[1] 8650356

In [25]:
saveRDS(final_dataset, "all_df_filtered_20260406.rds")

## Preprocessing

In [2]:
final_dataset <- readRDS("all_df_filtered_20260406.rds")

In [3]:
colnames(final_dataset)

[1] "Article_ID"   "Title"        "Date"         "Abstract"     "Text"        
 [6] "Source"       "Location"     "People"       "Organization" "Type"        
[11] "Desk"

In [4]:
table(final_dataset$Source)


                         New York Times                     The Washington Post 
                                3505186                                 1756782 
           The Washington Post  (1974-)            The Washington Post (Online) 
                                 409456                                  669707 
The Washington Post (pre-1997 Fulltext)                     Wall Street Journal 
                                 508783                                 1434011 
           Wall Street Journal  (1923-)                 Washington Post – Blogs 
                                 112406                                  254025 

### Sample

In [28]:
set.seed(123)

sampled_data <- final_dataset %>%
  filter(!Source %in% c("The Washington Post (Online)", "Washington Post – Blogs")) %>%
  mutate(Source = case_when(Source %in% c("The Washington Post", "The Washington Post  (1974-)", 
                                           "The Washington Post (pre-1997 Fulltext)") ~ "The Washington Post",
                           Source %in% c(" Wall Street Journal", "Wall Street Journal  (1923-)") ~ "Wall Street Journal",
                           TRUE ~ Source)) %>%
  mutate(Date = ymd(Date),
         Year = year(Date)) %>%
  group_by(Source, Year) %>%
  slice_sample(n = 10000) %>% 
  ungroup()

In [29]:
sampled_data <- sampled_data %>% filter(Year <= 2024)

nrow(sampled_data)

[1] 1350000

In [15]:
saveRDS(sampled_data, "26-04-12-sampled_data.rds")

In [3]:
## Check type distribution
library(stringr)
sampled_data_type <- 
    sampled_data %>%
    mutate(Type_New = case_when(str_detect(Type, "Commentary|Editorial|Review") ~ "Editorial",
                                str_detect(Type, "News|Article|Feature") ~ "News",
                                TRUE ~ "Other"))

In [4]:
sampled_data_type %>%
    group_by(Year) %>%
    summarise(Editorial_Percent = sum(Type_New == "Editorial")/n()) %>%
    ungroup()

Year,Editorial_Percent
<dbl>,<dbl>
1980,0.05473333
1981,0.04833333
1982,0.05343333
1983,0.04100000
1984,0.04450000
1985,0.04986667
1986,0.03690000
1987,0.02683333
1988,0.02493333


### Tokenize

In [16]:
corpus <- corpus(sampled_data, text_field = "Text")
  
all_toks <- tokens(
    corpus,
    remove_punct = TRUE,
    remove_symbols=TRUE, 
    remove_numbers=TRUE, 
    remove_separators=TRUE
  ) %>%
    tokens_tolower()

Warning message:
“NA is replaced by empty string”


In [17]:
# Remove all the 's
types_withs_all <- str_subset(types(all_toks), "(?<=[A-Za-z]{2,30})\\'s")
all_toks_nos <- tokens_replace(all_toks, types_withs_all, gsub("\\'s", "", types_withs_all))

In [18]:
saveRDS(all_toks_nos, "all_toks_nos.rds")

In [19]:
require(quanteda.textstats)

Loading required package: quanteda.textstats



In [20]:
#### Extract bi-grams #### 
set.seed(123)
all_toks_sample <- 
  tokens_sample(all_toks_nos, size=50000)

tstat_col <-  all_toks_sample %>%  
              textstat_collocations(min_count = 25)
              
saveRDS(tstat_col, "tstat_col_0412.rds")

In [21]:
library(SnowballC)

pattern <- paste0("\\b(", paste(c(stopwords("en"), "p\\.m", "pm", "a\\.m", "am", "block", "st", "mr"), 
                                collapse = "|"), 
                  ")\\b")

tstat_col_filter <-
    tstat_col %>%
    filter(z>=50 & lambda >=5) %>%
    filter(!str_detect(collocation, pattern))

write.csv(tstat_col_filter, "tstat_col_filter.csv")

In [22]:
lib_con_ngram <- c("liberal democratic party", "progressive conservative party", "liberal front party", 
                   "british conservative", "britain conservative", "conservative party",
                   "liberal arts", "neo liberalism", "liberal interpretation", "liberal definition",
                   "liberal use", "liberal lending", "liberal application", "liberal dress code",
                   "liberal return policy", "liberal return policies", "liberal pricing",
                   "liberal estimate", "liberal estimates", "liberal portion", "liberal dose", 
                   "liberal amount", "liberal amounts", "liberal heap", "liberal splash", "sauce liberal",
                   "liberal substitution", "liberal scoring",
                   "conservative investment", "conservative investments", "conservative investor", "conservative investors",
                   "conservative management", "conservative accounting", "conservative lending",
                   "conservative portfolio", "conservative portfolios",
                   "conservative estimate", "conservative estimates",  "conservative projection", "conservative projections", 
                   "conservative forecast", "conservative forecasts", "conservative forecaster", "conservative forecasters",
                   "conservative amount", "conservative amounts", "conservative bids",
                   "conservative assumptions", "conservative assumption", "conservative definition",
                   "conservative color", "conservative colors", "conservative look", "conservative outfit",
                   "conservative wardrobe", "conservative clothing", "conservative styling", 
                   "conservative suit", "conservative suits", "conservative dress", "conservative attire", 
                   "conservative mode",  "conservative painting", "conservative paintings", "conservative sofas",
                   "conservative contemporary", "conservative musical", "conservative harmonic", 
                   "conservative repertory", "conservative repertoire",  "conservative idiom",
                   "conservative production", "conservative productions", 
                   "conservative play", "conservative play-calling", "conservative coach", "conservative coaching",
                   "conservative passing", "conservative shot", "conservative shots", "conservative game",
                   "conservative composer", "conservative orchestra", "conservative houses",
                   "aesthetically conservative", "musically conservative", "artistically conservative", 
                   "financially conservative", "stylistically conservative", "harmonic conservatism")

In [23]:
ngrams_all <-
  c(tolower(tstat_col_filter$collocation),
    lib_con_ngram)

## concatenate

all_toks_ngram <-
  all_toks_nos %>%
  tokens_tolower() %>%
  tokens_compound(pattern = phrase(ngrams_all),
                  concatenator = "-")

In [25]:
saveRDS(all_toks_ngram, "all_toks_ngram_0412.rds")